# Imports

In [130]:
# Performing all imports
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler

# Creating all the required datasets

In [131]:
water_quality_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\water_quality_training_dataset.csv')
terraclimate_features_training = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\terraclimate_features_training.csv')
landsat_features_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\landsat_features_training.csv')

water_quality_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\submission_template.csv')
terraclimate_features_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\terraclimate_features_validation.csv')
landsat_features_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\landsat_features_validation.csv')

<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
C:\Users\SW559CS\AppData\Local\Temp\ipykernel_42648\1608632834.py:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
  water_quality_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\water_quality_training_dataset.csv')
C:\Users\SW559CS\AppData\Local\Temp\ipykernel_42648\16086328

# Combining Datasets

In [132]:
# Combining training datasets
water_terras_df = water_quality_training.merge(
    terraclimate_features_training,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge = water_terras_df.merge(
    landsat_features_training,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)


# Combining testing datasets
water_terras_testing_df = water_quality_testing.merge(
    terraclimate_features_testing,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge_testing = water_terras_testing_df.merge(
    landsat_features_testing,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge_testing.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'], inplace= True)
water_terras_testing_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'], inplace= True)

In [133]:
display(complete_merge)

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.20000,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.10000,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.50000,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.70000,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.20000,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9314,-27.527500,30.858056,23-12-2015,38.900,134.0,20.0,166.30000,15296.5,10043.0,16381.0,14443.0,-0.034236,-0.239858
9315,-26.861111,28.884722,23-12-2015,115.800,388.0,20.0,182.40001,15642.5,10294.5,17045.5,14710.0,-0.042921,-0.246928
9316,-26.984722,26.632278,23-12-2015,104.874,835.0,148.0,207.80000,14945.0,10732.0,18303.0,16281.0,-0.100999,-0.260754
9317,-27.935000,26.126667,23-12-2015,128.000,305.0,28.0,222.80000,14727.5,11051.0,18420.0,15724.5,-0.111396,-0.250042


# Data Cleaning And Feature Engineering for Train Data

In [134]:
# Filling in any missing data
complete_merge = complete_merge.fillna(complete_merge.median(numeric_only=True))
complete_merge.isna().sum()
# complete_merge = complete_merge[['swir22','NDMI','MNDWI','pet', 'nir', 'green', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

# Creating NVDI (Vegetation Index)
complete_merge['NVDI'] = (complete_merge['nir'] - complete_merge['green'])/(complete_merge['nir'] + complete_merge['green'])

# NDWI (McFeesters Water Index)
complete_merge['NDWI'] = (complete_merge['green'] - complete_merge['nir'])/(complete_merge['nir'] + complete_merge['green'])

#NSDI (Suspended sediments index)
complete_merge['NDSI'] = (complete_merge['swir22'] - complete_merge['green'])/(complete_merge['swir22'] + complete_merge['green'])

# Adding lag features

complete_merge['PET_lag_7'] = complete_merge.groupby(['Latitude', 'Longitude'])['pet'].shift(7)
# complete_merge['PET_lag_14'] = complete_merge.groupby(['Latitude', 'Longitude'])['pet'].shift(14)
# complete_merge['PET_lag_30'] = complete_merge.groupby(['Latitude', 'Longitude'])['pet'].shift(30)

# Getting Temporal features
complete_merge['Sample Date'] = pd.to_datetime(complete_merge['Sample Date'], format="%d-%m-%Y")
complete_merge['year'] = complete_merge['Sample Date'].dt.year
complete_merge['month'] = complete_merge['Sample Date'].dt.month
complete_merge['day'] = complete_merge['Sample Date'].dt.day
complete_merge['day_of_year'] = complete_merge['Sample Date'].dt.dayofyear
complete_merge['week_of_year'] = complete_merge['Sample Date'].dt.isocalendar().week.astype(int)
complete_merge['day_of_week'] = complete_merge['Sample Date'].dt.dayofweek       # Monday = 0
complete_merge['is_weekend'] = complete_merge['Sample Date'].isin([5, 6]).astype(int)

# Getting the Season

def get_season(month):
    if month in [12, 1, 2]:
        return 1
    elif month in [3, 4, 5]:
        return 2
    elif month in [6, 7, 8]:
        return 3
    else:
        return 4

complete_merge['season'] = complete_merge['month'].apply(get_season)

# Gathering relevant columns
# complete_merge = complete_merge.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
display(complete_merge)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,nir,green,swir16,...,NDSI,PET_lag_7,year,month,day,day_of_year,week_of_year,day_of_week,is_weekend,season
0,-28.760833,17.730278,2011-01-02,128.912,555.0,10.0,174.20000,11190.0,11426.0,7687.5,...,-0.198259,NaN,2011,1,2,2,52,6,0,1
1,-26.861111,28.884722,2011-01-03,74.720,162.9,163.0,124.10000,17658.5,9550.0,13746.5,...,0.050885,NaN,2011,1,3,3,1,0,0,1
2,-26.450000,28.085833,2011-01-03,89.254,573.0,80.0,127.50000,15210.0,10720.0,17974.0,...,0.139681,NaN,2011,1,3,3,1,0,0,1
3,-27.671111,27.236944,2011-01-03,82.000,203.6,101.0,129.70000,14887.0,10943.0,13522.0,...,0.020585,NaN,2011,1,3,3,1,0,0,1
4,-27.356667,27.286389,2011-01-03,56.100,145.1,151.0,129.20000,16828.5,9502.5,12665.5,...,0.007339,NaN,2011,1,3,3,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9314,-27.527500,30.858056,2015-12-23,38.900,134.0,20.0,166.30000,15296.5,10043.0,16381.0,...,0.179695,166.30000,2015,12,23,357,52,2,0,1
9315,-26.861111,28.884722,2015-12-23,115.800,388.0,20.0,182.40001,15642.5,10294.5,17045.5,...,0.176588,182.40001,2015,12,23,357,52,2,0,1
9316,-26.984722,26.632278,2015-12-23,104.874,835.0,148.0,207.80000,14945.0,10732.0,18303.0,...,0.205420,207.80000,2015,12,23,357,52,2,0,1
9317,-27.935000,26.126667,2015-12-23,128.000,305.0,28.0,222.80000,14727.5,11051.0,18420.0,...,0.174544,222.80000,2015,12,23,357,52,2,0,1


# Data Cleaning And Feature Engineering for Testing Data

In [135]:
# Filling in any missing data
complete_merge_testing = complete_merge_testing.fillna(complete_merge_testing.median(numeric_only=True))
complete_merge_testing.isna().sum()
# complete_merge = complete_merge[['swir22','NDMI','MNDWI','pet', 'nir', 'green', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

# Creating NVDI (Vegetation Index)
complete_merge_testing['NVDI'] = (complete_merge_testing['nir'] - complete_merge_testing['green'])/(complete_merge_testing['nir'] + complete_merge_testing['green'])

# NDWI (McFeesters Water Index)
complete_merge_testing['NDWI'] = (complete_merge_testing['green'] - complete_merge_testing['nir'])/(complete_merge_testing['nir'] + complete_merge_testing['green'])

#NSDI (Suspended sediments index)
complete_merge_testing['NDSI'] = (complete_merge_testing['swir22'] - complete_merge_testing['green'])/(complete_merge_testing['swir22'] + complete_merge_testing['green'])

# Adding lag features

complete_merge_testing['PET_lag_7'] = complete_merge_testing.groupby(['Latitude', 'Longitude'])['pet'].shift(7)
# complete_merge_testing['PET_lag_14'] = complete_merge_testing.groupby(['Latitude', 'Longitude'])['pet'].shift(14)
# complete_merge_testing['PET_lag_30'] = complete_merge_testing.groupby(['Latitude', 'Longitude'])['pet'].shift(30)

# Getting Temporal features
complete_merge_testing['Sample Date'] = pd.to_datetime(complete_merge_testing['Sample Date'], format="%d-%m-%Y")
complete_merge_testing['year'] = complete_merge_testing['Sample Date'].dt.year
complete_merge_testing['month'] = complete_merge_testing['Sample Date'].dt.month
complete_merge_testing['day'] = complete_merge_testing['Sample Date'].dt.day
complete_merge_testing['day_of_year'] = complete_merge_testing['Sample Date'].dt.dayofyear
complete_merge_testing['week_of_year'] = complete_merge_testing['Sample Date'].dt.isocalendar().week.astype(int)
complete_merge_testing['day_of_week'] = complete_merge_testing['Sample Date'].dt.dayofweek       # Monday = 0
complete_merge_testing['is_weekend'] = complete_merge_testing['Sample Date'].isin([5, 6]).astype(int)

# Getting the Season

def get_season(month):
    if month in [12, 1, 2]:
        return 1
    elif month in [3, 4, 5]:
        return 2
    elif month in [6, 7, 8]:
        return 3
    else:
        return 4

complete_merge_testing['season'] = complete_merge_testing['month'].apply(get_season)

# Gathering relevant columns
# complete_merge_testing = complete_merge_testing.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
display(complete_merge_testing)


,Latitude,Longitude,Sample Date,pet,nir,green,swir16,swir22,NDMI,MNDWI,...,NDSI,PET_lag_7,year,month,day,day_of_year,week_of_year,day_of_week,is_weekend,season
0,-32.043333,27.822778,2014-09-01,161.900010,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,...,-0.017676,NaN,2014,9,1,244,36,0,0,4
1,-33.329167,26.077500,2015-09-16,177.600000,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,...,0.024632,NaN,2015,9,16,259,38,2,0,4
2,-32.991639,27.640028,2015-05-07,158.400010,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,...,0.033926,NaN,2015,5,7,127,19,3,0,2
3,-34.096389,24.439167,2012-02-07,130.000000,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,...,0.024632,NaN,2012,2,7,38,6,1,0,1
4,-32.000556,28.581667,2014-10-01,152.500000,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,...,-0.120612,NaN,2014,10,1,274,40,2,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,-33.771111,25.386667,2012-12-06,171.400010,17562.0,9492.0,13559.5,10235.0,0.128609,-0.176453,...,0.037664,NaN,2012,12,6,341,49,3,0,1
196,-33.185361,27.390750,2014-09-04,159.400010,15883.0,9083.5,12135.5,9484.0,0.133751,-0.143833,...,0.021570,159.40001,2014,9,4,247,36,3,0,4
197,-32.043333,27.822778,2015-09-28,168.600000,13619.5,10046.5,13105.0,10969.0,0.019252,-0.132108,...,0.043896,159.70000,2015,9,28,271,40,0,0,4
198,-33.001667,25.161389,2015-01-08,81.200005,13955.5,10670.0,17303.5,14835.5,-0.107105,-0.237135,...,0.163318,NaN,2015,1,8,8,2,3,0,1


# Train/Test Split and training the Model

In [136]:
X = complete_merge.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Total Alkalinity']

cols_to_scale = ['pet','nir','green','swir16','swir22','NDMI','MNDWI', 'NVDI', 'NDWI', 'NDSI', 'PET_lag_7']

X_test = complete_merge_testing.drop(columns=['Latitude', 'Longitude', 'Sample Date'])

scaler = StandardScaler()
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(complete_merge_testing[cols_to_scale])

X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
r2 = r2_score(y_val, val_preds)
print(f'R2 Score: {r2}')

preds_ac = model.predict(X_test)



R2 Score: 0.5945851717115964


In [ ]:
X = complete_merge.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Electrical Conductance']

cols_to_scale = ['pet','nir','green','swir16','swir22','NDMI','MNDWI', 'NVDI', 'NDWI', 'NDSI', 'PET_lag_7']

X_test = complete_merge_testing.drop(columns=['Latitude', 'Longitude', 'Sample Date'])

scaler = StandardScaler()
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(complete_merge_testing[cols_to_scale])

X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
r2 = r2_score(y_val, val_preds)
print(f'R2 Score: {r2}')

preds_ec = model.predict(X_test)




R2 Score: 0.6170639912194968


In [138]:
X = complete_merge.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Dissolved Reactive Phosphorus']

cols_to_scale = ['pet','nir','green','swir16','swir22','NDMI','MNDWI', 'NVDI', 'NDWI', 'NDSI', 'PET_lag_7']

X_test = complete_merge_testing.drop(columns=['Latitude', 'Longitude', 'Sample Date'])

scaler = StandardScaler()
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(complete_merge_testing[cols_to_scale])

X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
r2 = r2_score(y_val, val_preds)
print(f'R2 Score: {r2}')

preds_drp= model.predict(X_test)


R2 Score: 0.49859543882297364


In [139]:
preds_ac_ec_drp_df = pd.DataFrame({
    'Total Alkalinity' : preds_ac,
    'Electrical Conductance' : preds_ec,
    'Dissolved Reactive Phosphorus': preds_drp
})

preds_ac_ec_drp_df.to_csv(r'C:\EY AI & Data Challenge\Datasets\Data_Manipulation\Preds.csv')